# Cơ sở lý thuyết và chiến lược tiền xử lý chuyên sâu dựa trên EDA

## 1) Kết quả EDA
Từ notebook EDA, ta đã thấy:
- Dữ liệu mất cân bằng lớp rất mạnh (imbalance ratio cao).
- Nhiều biến khoảng cách lệch phải và có outlier.
- `Elevation` có tín hiệu mạnh với target.
- Nhóm biến one-hot (`Wilderness_Area`, `Soil_Type`) chứa tín hiệu phân lớp rõ.

Vì vậy chiến lược tiền xử lý trong notebook này sẽ **giữ lại tín hiệu EDA quan trọng**, đồng thời tối ưu tốc độ.

## 2) Feature Engineering
Trong Covtype, biến khoảng cách ngang và dọc đến thủy văn mô tả vị trí tương đối nhưng chưa phản ánh trực tiếp khoảng cách hình học thực tế. Vì vậy ta tạo:

$$
\text{Euclidean\_Distance\_To\_Hydrology} = \sqrt{\text{Horizontal\_Distance\_To\_Hydrology}^2 + \text{Vertical\_Distance\_To\_Hydrology}^2}
$$

Ngoài ra, tạo biến tổng hợp mức độ xa tiện ích địa lý:

$$
\text{Distance\_To\_Amenities} = \frac{\text{Horizontal\_Distance\_To\_Roadways} + \text{Horizontal\_Distance\_To\_Fire\_Points} + \text{Euclidean\_Distance\_To\_Hydrology}}{3}
$$

Hai biến mới này giúp mô hình nhận diện logic không gian tốt hơn.

## 3) Outliers và Skewness
Các biến khoảng cách thường lệch phải (right-skew) và có outlier. Hướng xử lý:
- Đo skewness cho nhóm continuous.
- Dùng RobustScaler cho nhóm biến liên tục để giảm tác động outlier.
- Log transform chỉ là tùy chọn mở rộng nếu cần thử nghiệm thêm.

## 4) Chiến lược cân bằng lớp 
Có thể dùng SMOTETomek nhưng vì thời gian chạy khá lâu nên ta đổi sang dùng **class weighting**:
- Tính trọng số lớp từ `y_train` bằng `compute_class_weight`.
- Truyền `class_weight` hoặc `sample_weight` vào mô hình khi huấn luyện.

Lợi ích:
- Nhanh hơn rõ rệt trên dữ liệu lớn.
- Không làm tăng kích thước dữ liệu train.
- Hạn chế rủi ro sinh mẫu nhân tạo chưa tối ưu.

## 5) Data Pipeline với ColumnTransformer
Ta áp dụng phép biến đổi khác nhau cho từng nhóm biến:
- Nhóm continuous (10 biến gốc + biến mới): scale.
- Nhóm one-hot (`Wilderness_Area`, `Soil_Type`): giữ nguyên.

Dùng ColumnTransformer + Pipeline giúp chuẩn hóa quy trình và ngăn rò rỉ dữ liệu khi tách train/test.

## Bước 1: Tải dữ liệu và xác định nhóm biến

Mục tiêu:
- Tải dữ liệu raw từ đường dẫn tương đối.
- Tách biến mục tiêu và nhóm đặc trưng (continuous, wilderness, soil).
- Chuẩn bị metadata để dùng xuyên suốt các bước tiền xử lý.

In [1]:
# Bước 1 - Import thư viện và tải dữ liệu

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import FunctionTransformer, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_class_weight

# Đường dẫn tương đối từ thư mục code/Part2_Classification/data/
csv_path = "../../../data/raw/classification/covtype.csv"

df = pd.read_csv(csv_path)

# Xác định cột mục tiêu linh hoạt
if "Cover_Type" in df.columns:
    target_col = "Cover_Type"
elif "target" in df.columns:
    target_col = "target"
else:
    raise ValueError("Không tìm thấy cột mục tiêu. Kỳ vọng Cover_Type hoặc target.")

# Tách X, y
X = df.drop(columns=[target_col]).copy()
y = df[target_col].copy()

# Xác định nhóm cột
continuous_cols = X.columns[:10].tolist()
wilderness_cols = [c for c in X.columns if c.startswith("Wilderness_Area")]
soil_cols = [c for c in X.columns if c.startswith("Soil_Type")]
binary_cols = wilderness_cols + soil_cols

print("Kích thước dữ liệu gốc:", df.shape)
print("Số lớp:", y.nunique())
print("Số biến continuous:", len(continuous_cols))
print("Số biến binary:", len(binary_cols))

Kích thước dữ liệu gốc: (581012, 55)
Số lớp: 7
Số biến continuous: 10
Số biến binary: 44


## Kết quả

- Dữ liệu đã được tách thành `X` và `y`, sẵn sàng cho train/test split.
- Nhóm biến được phân rõ theo vai trò biến đổi:
  - Continuous: scale và kiểm tra skewness.
  - Binary one-hot: passthrough.
- Cách tách nhóm này là nền cho ColumnTransformer ở bước pipeline.

## Bước 2: Feature Engineering có định hướng địa lý

Mục tiêu:
- Tạo biến khoảng cách Euclid tới thủy văn để phản ánh khoảng cách không gian thực.
- Tạo biến tổng hợp Distance_To_Amenities để mô tả độ xa các hạ tầng chính.
- Đảm bảo quá trình tạo biến nằm trong pipeline bằng FunctionTransformer.

In [2]:
# Bước 2 - Feature Engineering bằng FunctionTransformer

def add_engineered_features(X_input: pd.DataFrame) -> pd.DataFrame:
    """Tạo đặc trưng mới từ nhóm khoảng cách địa lý."""
    X_out = X_input.copy()

    # Khoảng cách Euclid đến thủy văn
    X_out["Euclidean_Distance_To_Hydrology"] = np.sqrt(
        X_out["Horizontal_Distance_To_Hydrology"] ** 2
        + X_out["Vertical_Distance_To_Hydrology"] ** 2
    )

    # Khoảng cách trung bình tới các tiện ích địa lý quan trọng
    X_out["Distance_To_Amenities"] = (
        X_out["Horizontal_Distance_To_Roadways"]
        + X_out["Horizontal_Distance_To_Fire_Points"]
        + X_out["Euclidean_Distance_To_Hydrology"]
    ) / 3.0

    return X_out

feature_engineer = FunctionTransformer(add_engineered_features, validate=False)

# Tạo thử để kiểm tra nhanh
X_fe_preview = feature_engineer.transform(X)
new_feature_cols = ["Euclidean_Distance_To_Hydrology", "Distance_To_Amenities"]

print("Số cột trước FE:", X.shape[1])
print("Số cột sau FE:", X_fe_preview.shape[1])
print("Các cột mới:", new_feature_cols)
X_fe_preview[new_feature_cols].describe().T

Số cột trước FE: 54
Số cột sau FE: 56
Các cột mới: ['Euclidean_Distance_To_Hydrology', 'Distance_To_Amenities']


,count,mean,std,min,25%,50%,75%,max
Euclidean_Distance_To_Hydrology,581012.0,276.065482,217.047653,0.000000,108.461975,229.477668,393.814677,1418.916840
Distance_To_Amenities,581012.0,1535.501107,793.748802,34.757296,937.729775,1400.388525,1973.222112,4383.920287


## Kết quả

- Hai đặc trưng mới đã bổ sung thông tin hình học và thông tin tổng hợp tiện ích.
- `Euclidean_Distance_To_Hydrology` giúp mô hình hiểu khoảng cách thực thay vì chỉ nhìn riêng chiều ngang/dọc.
- `Distance_To_Amenities` làm mượt tín hiệu vị trí tổng thể, có thể hỗ trợ giảm nhiễu cục bộ từ từng biến đơn lẻ.

## Bước 3: Kiểm tra skewness và định hướng xử lý outlier

Mục tiêu:
- Đo độ lệch phân phối của các biến continuous (gốc + mới).
- Xác định biến nào lệch mạnh để cân nhắc log transform ở vòng thử nghiệm tiếp theo.
- Chọn RobustScaler để giảm ảnh hưởng outlier ngay trong pipeline chuẩn.

In [3]:
# Bước 3 - Kiểm tra skewness trước khi scale

continuous_plus_new = continuous_cols + ["Euclidean_Distance_To_Hydrology", "Distance_To_Amenities"]

skew_values = X_fe_preview[continuous_plus_new].skew().sort_values(key=np.abs, ascending=False)
print("Skewness của nhóm continuous + biến mới:")
print(skew_values)

high_skew = skew_values[np.abs(skew_values) > 1.0]
print("\nCác biến có |skew| > 1.0 (đáng chú ý):")
print(high_skew if len(high_skew) > 0 else "Không có biến vượt ngưỡng 1.0")

Skewness của nhóm continuous + biến mới:
Vertical_Distance_To_Hydrology        1.790250
Horizontal_Distance_To_Fire_Points    1.288644
Hillshade_9am                        -1.181147
Horizontal_Distance_To_Hydrology      1.140437
Euclidean_Distance_To_Hydrology       1.133469
Hillshade_Noon                       -1.063056
Elevation                            -0.817596
Distance_To_Amenities                 0.805075
Slope                                 0.789273
Horizontal_Distance_To_Roadways       0.713679
Aspect                                0.402628
Hillshade_3pm                        -0.277053
dtype: float64

Các biến có |skew| > 1.0 (đáng chú ý):
Vertical_Distance_To_Hydrology        1.790250
Horizontal_Distance_To_Fire_Points    1.288644
Hillshade_9am                        -1.181147
Horizontal_Distance_To_Hydrology      1.140437
Euclidean_Distance_To_Hydrology       1.133469
Hillshade_Noon                       -1.063056
dtype: float64


## Diễn giải 

- Nếu nhóm biến khoảng cách có |skew| cao, RobustScaler là lựa chọn an toàn hơn StandardScaler.
- Mục tiêu ở bước này là giảm ảnh hưởng điểm cực trị mà vẫn giữ cấu trúc thông tin.
- Log transform được xem là tùy chọn nâng cao, chỉ nên bật khi cần sau khi so sánh mô hình.

## Bước 4: Chia Train/Test với stratify

Mục tiêu:
- Tách dữ liệu đánh giá công bằng theo tỉ lệ 80/20.
- Dùng `stratify=y` để bảo toàn phân bố lớp giữa train và test.
- Tránh hiện tượng lớp hiếm bị thiếu ở tập test.

In [4]:
# Bước 4 - Chia dữ liệu train/test với stratify

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Kích thước X_train:", X_train_raw.shape)
print("Kích thước X_test:", X_test_raw.shape)
print("\nPhân bố y_train:")
print(y_train.value_counts(normalize=True).sort_index().round(4))
print("\nPhân bố y_test:")
print(y_test.value_counts(normalize=True).sort_index().round(4))

Kích thước X_train: (464809, 54)
Kích thước X_test: (116203, 54)

Phân bố y_train:
Cover_Type
1    0.3646
2    0.4876
3    0.0615
4    0.0047
5    0.0163
6    0.0299
7    0.0353
Name: proportion, dtype: float64

Phân bố y_test:
Cover_Type
1    0.3646
2    0.4876
3    0.0615
4    0.0047
5    0.0163
6    0.0299
7    0.0353
Name: proportion, dtype: float64


## Diễn giải 

- Phân bố lớp giữa train và test gần tương đương nhau nhờ `stratify`.
- Điều này giúp kết quả đánh giá cuối phản ánh đúng độ khó đa lớp của dữ liệu gốc.
- Đây là điều kiện cần trước khi áp dụng resampling chỉ trên train.

## Bước 5: Xây dựng Pipeline tiền xử lý bằng ColumnTransformer

Mục tiêu:
- Đưa toàn bộ phép biến đổi về một pipeline nhất quán.
- Áp dụng `RobustScaler` cho nhóm continuous + biến mới.
- Giữ nguyên nhóm one-hot bằng `passthrough`.

In [5]:
# Bước 5 - Pipeline tiền xử lý

# Danh sách continuous mở rộng sau khi tạo đặc trưng
continuous_features_final = continuous_cols + [
    "Euclidean_Distance_To_Hydrology",
    "Distance_To_Amenities",
]

# Pipeline cho nhóm continuous
continuous_pipeline = Pipeline(
    steps=[
        ("scaler", RobustScaler()),
    ]
)

# ColumnTransformer kết hợp biến đổi theo nhóm cột
preprocessor = ColumnTransformer(
    transformers=[
        ("continuous", continuous_pipeline, continuous_features_final),
        ("binary", "passthrough", binary_cols),
    ],
    remainder="drop",
)

# Pipeline đầy đủ: FE -> ColumnTransformer
preprocess_pipeline = Pipeline(
    steps=[
        ("feature_engineering", feature_engineer),
        ("column_transform", preprocessor),
    ]
)

# Fit trên train và transform train/test để tránh leakage
X_train_processed = preprocess_pipeline.fit_transform(X_train_raw)
X_test_processed = preprocess_pipeline.transform(X_test_raw)

print("Kích thước X_train sau preprocess:", X_train_processed.shape)
print("Kích thước X_test sau preprocess:", X_test_processed.shape)

Kích thước X_train sau preprocess: (464809, 56)
Kích thước X_test sau preprocess: (116203, 56)


## Diễn giải 

- Pipeline áp dụng đúng nguyên tắc chống leakage: fit trên train, transform trên test.
- Bám sát EDA:
  - Giữ và scale nhóm continuous (có skew/outlier).
  - Giữ nguyên one-hot (`Wilderness_Area`, `Soil_Type`) vì EDA cho thấy nhóm này mang tín hiệu phân lớp rõ.
- Sau bước này dữ liệu đã sẵn sàng cho chiến lược cân bằng lớp theo hướng nhẹ và nhanh.

## Bước 6: Cân bằng lớp theo hướng chạy nhanh (Class Weight)

Mục tiêu:
- Không resample dữ liệu train để tránh tăng thời gian xử lý.
- Tính trọng số lớp từ `y_train` nhằm bù mất cân bằng.
- Chuẩn bị `class_weight_dict` và `sample_weight_train` để dùng trực tiếp khi fit model.

In [ ]:
# Bước 6 - Cân bằng lớp bằng class weighting (nhanh hơn resampling)

# Tính class weight trên tập train gốc (sau split)
classes = np.sort(y_train.unique())
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight_dict = {int(cls): float(w) for cls, w in zip(classes, weights)}

# Tạo sample weight cho từng dòng train (hữu ích cho nhiều mô hình hỗ trợ sample_weight)
sample_weight_train = y_train.map(class_weight_dict).to_numpy()

print("Class weights (dựa trên y_train):")
print(class_weight_dict)
print("\nThống kê sample_weight_train:")
print("Min:", sample_weight_train.min())
print("Max:", sample_weight_train.max())
print("Mean:", sample_weight_train.mean())

print("\nKích thước dữ liệu train sau bước này (không đổi):")
print("X_train_processed:", X_train_processed.shape)
print("y_train:", y_train.shape)

print("\nPhân bố lớp train để đối chiếu mất cân bằng:")
print(y_train.value_counts().sort_index())


Class weights (dựa trên y_train):
{1: 0.39181272253992233, 2: 0.29298131712974634, 3: 2.3214797648598298, 4: 30.209866112049916, 5: 8.743914368486399, 6: 4.779133850171708, 7: 4.046884794873581}

Thống kê sample_weight_train:
Min: 0.29298131712974634
Max: 30.209866112049916
Mean: 1.0

Kích thước dữ liệu train sau bước này (không đổi):
X_train_processed: (464809, 56)
y_train: (464809,)

Phân bố lớp train để đối chiếu mất cân bằng:
Cover_Type
1    169472
2    226640
3     28603
4      2198
5      7594
6     13894
7     16408
Name: count, dtype: int64


## Diễn giải 

- Class weighting giúp tăng "độ quan tâm" của mô hình với lớp thiểu số mà không cần sinh thêm dữ liệu.
- So với SMOTETomek, cách này chạy nhanh hơn đáng kể trên Covtype do không thực hiện nội suy mẫu và làm sạch biên.
- Kích thước tập train giữ nguyên, nên tiết kiệm cả thời gian fit và bộ nhớ.
- Ở bước huấn luyện, chỉ cần truyền `class_weight_dict` hoặc `sample_weight_train` vào mô hình phù hợp.

# Tổng kết tiền xử lý 

## 1) Giữ gì theo EDA
- Giữ Feature Engineering địa lý: `Euclidean_Distance_To_Hydrology`, `Distance_To_Amenities`.
- Giữ RobustScaler cho nhóm continuous do skewness và outlier.
- Giữ nguyên nhóm one-hot (`Wilderness_Area`, `Soil_Type`) vì có tín hiệu phân lớp mạnh.
- Giữ train/test split có `stratify=y`.

## 2) Tối ưu tốc độ
- Bỏ SMOTETomek trong pipeline chính vì chi phí tính toán cao trên dữ liệu lớn.
- Thay bằng class weighting để xử lý mất cân bằng nhanh hơn, nhẹ hơn và ổn định hơn khi lặp nhiều thử nghiệm.

## 3) Tránh Data Leakage
- Mọi bước học tham số (scaler, biến đổi) được `fit` trên train.
- Tập test chỉ `transform`, không tham gia học tham số tiền xử lý.
- Cân bằng lớp bằng trọng số nên không cần can thiệp trực tiếp lên tập test.

## 4) Ảnh hưởng đến kích thước dữ liệu
- Với class weighting: kích thước train/test **không đổi**.
- Đổi lại, mô hình học có trọng số để không thiên lệch về lớp đa số.

## 5) Gợi ý mở rộng
- Nếu cần benchmark thêm: thử RandomUnderSampler (nhanh hơn SMOTETomek) như một baseline tốc độ.
- Đánh giá macro-F1, balanced accuracy, recall theo lớp để phản ánh đúng mục tiêu đa lớp mất cân bằng.